In [1]:
%pip install requests pandas pyproj

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from pyproj import Transformer

# ============================
# 1. 로컬 CSV 경로 지정 (형님 파일명으로 바꿔주세요)
# ============================
PATH_VDS   = "VDS_84_04_01_936124.csv"    # VDS 지점 교통량·속도 CSV
PATH_COORD = "ETC_S0_07_04_709280.csv"    # 도로중심선_이정_좌표 CSV
PATH_ZONE  = "ETC_79_04_01_990783.csv"       # VDS존 CSV

# ============================
# 2. 컬럼 이름 매핑 (파일 보고 수정 가능)
# ============================

# --- VDS 교통량 ---
COL_DATE      = "기준일"
COL_HOUR      = "기준시"
COL_TIMESEG   = "기준시간"
COL_VDS_ID    = "VDS_ID"
COL_VOLUME    = "교통량"
COL_SPEED     = "평균속도"
COL_ROUTE     = "노선번호"   # ← 필요하면 나중에 실제 컬럼명으로 변경
COL_DIST      = "도로이정"
COL_NODE_NAME = "노드명"

# --- 좌표 파일 ---
COL_COORD_ROUTE = "노선번호"
COL_COORD_DIST  = "이정"
COL_COORD_X     = "GRS80X좌표값"
COL_COORD_Y     = "GRS80Y좌표값"

# --- VDS존 ---
COL_ZONE_VDS_ID = "VDS_ID"
COL_ZONE_ID     = "콘존ID"     # 구간ID로 사용

# ============================
# 3. 좌표계 변환기
# ============================
transformer = Transformer.from_crs("EPSG:5181", "EPSG:4326", always_xy=True)

def add_latlon_from_xy(df, x_col, y_col, lon_col="경도", lat_col="위도"):
    x = pd.to_numeric(df[x_col], errors="coerce")
    y = pd.to_numeric(df[y_col], errors="coerce")
    lons, lats = transformer.transform(x.values, y.values)
    df[lon_col] = lons
    df[lat_col] = lats
    return df

# ============================
# 4. 메인 로직
# ============================

def main():
    # --- 1) 로컬 CSV 읽기 ---
    vds_df   = pd.read_csv(PATH_VDS,   engine="python", sep=None, encoding="cp949")
    coord_df = pd.read_csv(PATH_COORD, engine="python", sep=None, encoding="cp949")
    zone_df  = pd.read_csv(PATH_ZONE,  engine="python", sep=None, encoding="cp949")


    print("[VDS 컬럼]", vds_df.columns.tolist())
    print("[COORD 컬럼]", coord_df.columns.tolist())
    print("[ZONE 컬럼]", zone_df.columns.tolist())

    # --- 2) 노선번호/이정 형식 맞추기 ---
    vds_df[COL_ROUTE] = vds_df[COL_ROUTE].astype(str).str.strip()
    coord_df[COL_COORD_ROUTE] = coord_df[COL_COORD_ROUTE].astype(str).str.strip()

    vds_df[COL_DIST] = pd.to_numeric(vds_df[COL_DIST], errors="coerce")
    coord_df[COL_COORD_DIST] = pd.to_numeric(coord_df[COL_COORD_DIST], errors="coerce")

    # --- 3) VDS × 좌표 조인 ---
    merged = pd.merge(
        vds_df,
        coord_df,
        left_on=[COL_ROUTE, COL_DIST],
        right_on=[COL_COORD_ROUTE, COL_COORD_DIST],
        how="left"
    )
    print("[INFO] VDS + 좌표 조인:", merged.shape)

    # --- 4) 위도/경도 생성 ---
    merged = add_latlon_from_xy(merged, COL_COORD_X, COL_COORD_Y)

    # --- 5) VDS존 ID 붙이기 ---
    zone_df[COL_ZONE_VDS_ID] = zone_df[COL_ZONE_VDS_ID].astype(str).str.strip()
    merged[COL_VDS_ID]       = merged[COL_VDS_ID].astype(str).str.strip()

    zone_small = zone_df[[COL_ZONE_VDS_ID, COL_ZONE_ID]].drop_duplicates()

    merged = pd.merge(
        merged,
        zone_small,
        left_on=COL_VDS_ID,
        right_on=COL_ZONE_VDS_ID,
        how="left"
    )
    print("[INFO] VDS + ZONE 조인:", merged.shape)

    # --- 6) 시간 문자열 생성 ---
    merged["측정일시"] = (
        merged[COL_DATE].astype(str).str.strip()
        + " "
        + merged[COL_HOUR].astype(str).str.zfill(2)
        + ":"
        + merged[COL_TIMESEG].astype(str).str.zfill(2)
    )

    # --- 7) 최종 DF 정리 ---
    final_df = merged[[
        COL_ZONE_ID,      # 구간ID
        COL_VDS_ID,
        COL_NODE_NAME,
        "측정일시",
        COL_VOLUME,
        COL_SPEED,
        COL_ROUTE,
        COL_DIST,
        "경도",
        "위도",
    ]]

    final_df.rename(columns={
        COL_ZONE_ID: "구간ID",
        COL_VDS_ID: "VDS_ID",
        COL_NODE_NAME: "지점명",
        COL_ROUTE: "노선번호",
        COL_DIST: "이정"
    }, inplace=True)

    print("[최종 미리보기]")
    print(final_df.head())

    # 저장
    final_df.to_csv("traffic_latlon_final.csv", index=False, encoding="utf-8-sig")
    print("\n[완료] traffic_latlon_final.csv 저장 완료")

if __name__ == "__main__":
    main()


[VDS 컬럼] ['기준시간', '기준시', '기준일', 'VDS_ID', '요일명', '지점이정', '노드명', '도로이정', '노선번호', '도로명', '교통량', '평균속도', 'Unnamed: 12']
[COORD 컬럼] ['노선번호', '도로명', '이정', 'X좌표값', 'Y좌표값', 'GRS80X좌표값', 'GRS80Y좌표값']
[ZONE 컬럼] ['VDS_ID', '지점이정', 'VDS존시작이정', 'VDS존종료이정', '노선번호', 'VDS존유형구분코드', '노선구성순번', '기점종점방향구분코드', 'VDS존길이', '도로등급구분코드', '콘존ID']
[INFO] VDS + 좌표 조인: (759744, 19)
[INFO] VDS + ZONE 조인: (759744, 22)


C:\Users\runaw\AppData\Local\Temp\ipykernel_10880\2495644023.py:122: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.rename(columns={


[최종 미리보기]
         구간ID        VDS_ID   지점명            측정일시  교통량    평균속도 노선번호    이정  \
0  0010CZE010  0010VDE00100  구서IC  20251203 00:00   59   87.02   10  0.20   
1  0010CZE011  0010VDE00200  영락IC  20251203 00:00   59   98.61   10  2.02   
2  0010CZE011  0010VDE00300  영락IC  20251203 00:00   -1   -1.00   10  2.02   
3  0010CZE020  0010VDE00401  부산TG  20251203 00:00   34  102.62   10  4.01   
4  0010CZE030  0010VDE00500  노포IC  20251203 00:00   -1   -1.00   10  5.08   

           경도         위도  
0  129.118375  36.148587  
1         NaN        NaN  
2         NaN        NaN  
3         NaN        NaN  
4         NaN        NaN  

[완료] traffic_latlon_final.csv 저장 완료
